# Lab 1 · ItemKNN

The runnable companion of the lab page, `docs/labs/01-itemknn.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "itemknn"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
item = int(data.item_pop.argmax())  # the most popular item
print("neighbours of", data.item_text[item] or data.item_ids[item])
lab.neighbours(model, data, item)

**Exercise.** Find the neighbours of a **rare** item, one with 5 to 10 interactions. Do they look as sensible as the popular item's? Then fit the model again with `knn_shrink=0` and with `knn_shrink=500` and compare the rare item's neighbours.

<details><summary>Hint</summary>

`np.flatnonzero((data.item_pop >= 5) & (data.item_pop <= 10))` lists the rare items. `lab.fit(METHOD, data, **{**best, "knn_shrink": 0})` changes one setting.

</details>

<details><summary>Solution</summary>

```python
import numpy as np

rare = int(np.flatnonzero((data.item_pop >= 5) & (data.item_pop <= 10))[0])
for shrink in (0, 500):
    m = lab.fit(METHOD, data, **{**best, "knn_shrink": shrink})
    print("shrink", shrink)
    print(lab.neighbours(m, data, rare, k=5))
```
With `knn_shrink=0`, pairs seen together by a single user can reach a similarity near 1, so the rare item's list fills with other rare items. A large shrink pushes those small-evidence pairs down.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "knn_neighbors", [5, 15, 50, 200, 1000], start="best")
lab.plot_sweep(table)
table

**Your prediction for `knn_neighbors`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "train_window_days", [None, 30, 90, 365], start="best")
lab.plot_sweep(table)
table

**Your prediction for `train_window_days`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "train_window_keep_last", [10, 50, 200, 1000], start="best")
lab.plot_sweep(table)
table

**Your prediction for `train_window_keep_last`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh itemknn:no-time-knobs
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `no-time-knobs` is ready to run; `keep-more-history`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_itemknn_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:no-time-knobs")

In [ ]:
lab.compare(METHOD, f"{METHOD}:keep-more-history")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`asymmetric` in `labs/01-itemknn/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:asymmetric")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:asymmetric", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.